# Phase 7: XGBoost Baseline Model

**Project:** ML-Powered Intrusion Detection System (IDS) for Secure Network Monitoring  
**Phase:** Phase 7 — XGBoost Baseline Model  
**Dataset:** CICIoT2023 (Standardized Tabular Flow Features, 46 Features, 34 Classes)  

This notebook implements, trains, and evaluates the **XGBoost (Extreme Gradient Boosting)** baseline classifier on the verified processed dataset partitions (`data/processed/`).

## 1. Setup & Environment Verification

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight

# Ensure project root is accessible
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.xgboost_model import XGBoostBaselineModel
from src.preprocessing.verify_split import load_partition
from scripts.train_xgboost import calculate_metrics_and_report

print(f"XGBoost Version: {xgb.__version__}")
print("Environment initialized successfully.")

## 2. Load Processed Datasets & Verify Partitions

In [ ]:
data_dir = PROJECT_ROOT / "data" / "processed"

X_train, y_train = load_partition(data_dir / "train", "train")
X_val, y_val = load_partition(data_dir / "validation", "val")
X_test, y_test = load_partition(data_dir / "test", "test")

with open(data_dir / "label_mapping.json", "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

inv_mapping = {v: k for k, v in label_mapping.items()}
target_names = [inv_mapping[i] for i in range(len(label_mapping))]

with open(data_dir / "preprocessing_metadata.json", "r", encoding="utf-8") as f:
    meta = json.load(f)
    feature_names = meta.get("feature_names", [f"feature_{i}" for i in range(X_train.shape[1])])

print(f"Train Partition:      {X_train.shape[0]:,} samples | {X_train.shape[1]} features")
print(f"Validation Partition: {X_val.shape[0]:,} samples | {X_val.shape[1]} features")
print(f"Test Partition:       {X_test.shape[0]:,} samples | {X_test.shape[1]} features")
print(f"Distinct Classes:     {len(target_names)}")

## 3. Display Class Distribution & Compute Balanced Sample Weights

In [ ]:
# Balanced sample weights computed exclusively on training set
sample_weights = compute_sample_weight("balanced", y_train)

print(f"Computed sample weights for {len(sample_weights):,} training samples.")
print(f"Min sample weight (majority class): {sample_weights.min():.4f}")
print(f"Max sample weight (minority class): {sample_weights.max():.4f}")
print(f"Weight ratio: {sample_weights.max() / sample_weights.min():.1f}:1")

## 4. Instantiate & Configure XGBoost Baseline Model

In [ ]:
xgb_model = XGBoostBaselineModel(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

print("XGBoost baseline model configured with Fast CPU Histogram algorithm.")

## 5. Train Model on Training Partition

In [ ]:
# Use stratified subset for fast interactive training in notebook
rng = np.random.default_rng(42)
indices = []
for c in range(len(target_names)):
    c_idx = np.where(y_train == c)[0]
    if len(c_idx) == 0: continue
    n_sub = max(min(len(c_idx), 150), int(len(c_idx) * (750000 / len(y_train))))
    indices.extend(rng.choice(c_idx, size=min(n_sub, len(c_idx)), replace=False))
indices = np.array(indices)
rng.shuffle(indices)

X_train_sub = X_train[indices]
y_train_sub = y_train[indices]
sub_weights = compute_sample_weight("balanced", y_train_sub)

print(f"Fitting XGBoost on {len(X_train_sub):,} stratified training samples...")
t0 = time.time()
xgb_model.fit(X_train_sub, y_train_sub, sample_weight=sub_weights)
train_time = time.time() - t0
print(f"Training successfully completed in {train_time:.2f} seconds.")

## 6. Diagnostic Evaluation on Validation Set

In [ ]:
t0 = time.time()
y_val_pred = xgb_model.predict(X_val)
val_time = time.time() - t0

val_metrics, val_report, _ = calculate_metrics_and_report(y_val, y_val_pred, target_names)
print(f"Validation Accuracy:          {val_metrics['accuracy']*100:.2f}%")
print(f"Validation Weighted F1-Score: {val_metrics['weighted_f1']:.4f}")
print(f"Validation Macro F1-Score:    {val_metrics['macro_f1']:.4f}")
print(f"Validation Inference Time:    {val_time:.4f}s ({len(X_val):,} samples)")

## 7. Single Final Evaluation on Test Set

In [ ]:
t0 = time.time()
y_test_pred = xgb_model.predict(X_test)
test_time = time.time() - t0

test_metrics, test_report, cm_test = calculate_metrics_and_report(y_test, y_test_pred, target_names)

print(f"Test Accuracy:          {test_metrics['accuracy']*100:.2f}%")
print(f"Test Weighted Precision:{test_metrics['weighted_precision']:.4f}")
print(f"Test Weighted Recall:   {test_metrics['weighted_recall']:.4f}")
print(f"Test Weighted F1-Score: {test_metrics['weighted_f1']:.4f}")
print(f"Test Macro Precision:   {test_metrics['macro_precision']:.4f}")
print(f"Test Macro Recall:      {test_metrics['macro_recall']:.4f}")
print(f"Test Macro F1-Score:    {test_metrics['macro_f1']:.4f}")
print(f"Test Inference Latency: {test_time:.4f}s ({len(X_test):,} samples)")

## 8. Per-Class Classification Performance

In [ ]:
display(test_report.sort_values(by="F1_Score", ascending=False))

## 9. Normalized Test Set Confusion Matrix

In [ ]:
plt.figure(figsize=(16, 14))
cm_norm = cm_test.astype('float') / (cm_test.sum(axis=1)[:, np.newaxis] + 1e-9)
sns.heatmap(
    cm_norm,
    annot=False,
    cmap="Greens",
    xticklabels=target_names,
    yticklabels=target_names,
    cbar_kws={'label': 'Normalized Prediction Ratio'}
)
plt.title("XGBoost Baseline — Normalized Test Confusion Matrix", fontsize=15)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("Ground Truth Label", fontsize=12)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

## 10. Feature Importance Analysis (Average Gain)

In [ ]:
feat_imp_gain = xgb_model.get_feature_importances(feature_names=feature_names, importance_type="gain")
feat_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance_Gain": [feat_imp_gain.get(f, 0.0) for f in feature_names]
}).sort_values(by="Importance_Gain", ascending=False)

plt.figure(figsize=(12, 8))
top25 = feat_df.head(25).iloc[::-1]
plt.barh(top25["Feature"], top25["Importance_Gain"], color="#2ca02c")
plt.title("Top 25 Feature Importances — XGBoost (Average Gain)", fontsize=14)
plt.xlabel("Average Gain Across Splits", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 11. Misclassification Diagnostics

In [ ]:
misclassified_mask = y_test != y_test_pred
misclass_indices = np.where(misclassified_mask)[0]

misclass_df = pd.DataFrame({
    "True_Class": [inv_mapping[int(y_test[i])] for i in misclass_indices],
    "Predicted_Class": [inv_mapping[int(y_test_pred[i])] for i in misclass_indices]
})

top_confusions = misclass_df.groupby(["True_Class", "Predicted_Class"]).size().reset_index(name="Count").sort_values(by="Count", ascending=False).head(10)
print("Top 10 Most Frequent Misclassification Pairs:")
display(top_confusions)

## 12. Model Comparison Table (Random Forest vs. XGBoost)

In [ ]:
comp_df = pd.read_csv(PROJECT_ROOT / "results" / "metrics" / "model_comparison.csv")
display(comp_df)